# Resume Detector - Kelompok 5

### Library
---

In [20]:
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

### Import Data
---

In [21]:
DATASET_PATH = "Resume Dataset\Resume\Resume.csv"
def load_and_prepare_data(DATASET_PATH):
    print("[-] Load Data")
    df = pd.read_csv(DATASET_PATH)
    
    df = df.dropna(subset=['Resume_str', 'Category'])
    return df

### Preprocessing
---

In [22]:
def preprocess_data(df):
    print("[-] Preprocessing Data")
    
    df['cleaned_resume'] = df['Resume_str'].apply(lambda x: " ".join(str(x).split()))
    df['cleaned_category'] = df['Category'].apply(lambda x: str(x).strip())
    
    examples = []
    for _, row in df.iterrows():
        examples.append(InputExample(
            texts=[row['cleaned_resume'], row['cleaned_category']], 
            label=1.0
        ))
        
    return df, examples

In [23]:
if __name__ == "__main__":
    MODEL_NAME = "all-MiniLM-L6-V2"
    OUTPUT_DIR = "./exported_resume_transformer"

    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"Please place the dataset at: {DATASET_PATH}")

    # Load and Clean
    df = load_and_prepare_data(DATASET_PATH)
    df, all_examples = preprocess_data(df)

[-] Load Data
[-] Preprocessing Data


### Split and Train
---

In [26]:
print("[-] Splitting dataset into Train and Validation")

train_examples, val_examples = train_test_split(all_examples, test_size=0.15, random_state=42)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

[-] Splitting dataset into Train and Validation


### Modelling (all-MiniLM-L6-V2)
---

In [ ]:
print(f"[-] Initializing base model: {MODEL_NAME}")

model = SentenceTransformer(MODEL_NAME)

train_loss = losses.CosineSimilarityLoss(model)

### Evaluation
---

In [ ]:
print("[-] Setting up Evaluator...")

val_sentences1 = [ex.texts[0] for ex in val_examples]
val_sentences2 = [ex.texts[1] for ex in val_examples]
val_labels = [ex.label for ex in val_examples]

evaluator = EmbeddingSimilarityEvaluator(val_sentences1, val_sentences2, val_labels)

In [ ]:
print("[-] Fine-Tuning")
    
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    evaluation_steps=50,
    warmup_steps=100,
    output_path=os.path.join(OUTPUT_DIR, "checkpoint")
    )

### Export Model
---

In [ ]:
print(f"[-] Exporting Final Model - {OUTPUT_DIR}")

model.save(OUTPUT_DIR)

print("[+] Model successfully exported! Ready for your demo deployment.")